# Tutorial: SL-01 Bias Variance Demo

Audience:
- Learners studying introductory statistical learning theory.

Prerequisites:
- Probability basics, mean squared error, and linear regression.
- Familiarity with train/test splits and polynomial features.

Learning goals:
- Show empirical overfitting in polynomial regression.
- Compare training, test, and cross-validation error across model complexity.
- Estimate bias and variance from repeated resampling.


## Outline

1. Build a noisy regression problem with a known target function.
2. Sweep polynomial degree and compare train, test, and 5-fold CV error.
3. Inspect representative low-, medium-, and high-degree fits.
4. Estimate squared bias and variance by repeated sampling.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

SEED = 7
rng = np.random.default_rng(SEED)
SEED


## Step 1 - Generate a controlled regression problem

We use a sinusoidal target function so that the true signal is known. A small noisy sample makes it easy to see how a flexible polynomial can drive training error down while test and cross-validation error rise.


In [ ]:
def target_function(x: np.ndarray) -> np.ndarray:
    return np.sin(2.0 * np.pi * x)


def make_dataset(sample_size: int = 18, noise_std: float = 0.22, seed: int = SEED):
    local_rng = np.random.default_rng(seed)
    x_train = np.sort(local_rng.uniform(0.0, 1.0, size=sample_size)).reshape(-1, 1)
    y_train = target_function(x_train[:, 0]) + local_rng.normal(0.0, noise_std, size=sample_size)
    x_test = np.linspace(0.0, 1.0, 400).reshape(-1, 1)
    y_test = target_function(x_test[:, 0])
    return x_train, y_train, x_test, y_test


def degree_pipeline(degree: int) -> Pipeline:
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("regressor", LinearRegression()),
        ]
    )


x_train, y_train, x_test, y_test = make_dataset()

fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.scatter(x_train[:, 0], y_train, color="black", label="train sample", zorder=3)
ax.plot(x_test[:, 0], y_test, linestyle="--", color="tab:gray", label="true signal")
ax.set_title("Synthetic regression problem")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
fig.tight_layout()
plt.close(fig)

{"sample_size": len(x_train), "noise_std": 0.22}


## Step 2 - Compare train, test, and cross-validation error

Training error always rewards flexibility. Test error and cross-validation error are the quantities to watch when choosing model complexity.


In [ ]:
degrees = list(range(1, 13))
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
results = []

for degree in degrees:
    model = degree_pipeline(degree)
    model.fit(x_train, y_train)

    train_mse = mean_squared_error(y_train, model.predict(x_train))
    test_mse = mean_squared_error(y_test, model.predict(x_test))
    cv_mse = -cross_val_score(
        degree_pipeline(degree),
        x_train,
        y_train,
        cv=cv,
        scoring="neg_mean_squared_error",
    ).mean()

    results.append(
        {
            "degree": degree,
            "train_mse": float(train_mse),
            "test_mse": float(test_mse),
            "cv_mse": float(cv_mse),
        }
    )

fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.plot(degrees, [row["train_mse"] for row in results], marker="o", label="train MSE")
ax.plot(degrees, [row["test_mse"] for row in results], marker="o", label="test MSE")
ax.plot(degrees, [row["cv_mse"] for row in results], marker="o", label="5-fold CV MSE")
ax.set_title("Model complexity versus predictive error")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("Mean squared error")
ax.legend()
fig.tight_layout()
plt.close(fig)

sorted(results, key=lambda row: row["cv_mse"])[:5]


## Step 3 - Inspect fitted curves

A low-degree polynomial usually misses structure, while a high-degree polynomial bends toward the noise. A middle degree often balances the two.


In [ ]:
selected_degrees = [1, 5, 10]
models = {degree: degree_pipeline(degree).fit(x_train, y_train) for degree in selected_degrees}

fig, axes = plt.subplots(1, len(selected_degrees), figsize=(14, 3.8), sharey=True)

for ax, degree in zip(axes, selected_degrees):
    model = models[degree]
    ax.scatter(x_train[:, 0], y_train, color="black", s=18, zorder=3)
    ax.plot(x_test[:, 0], y_test, linestyle="--", color="tab:gray", label="true signal")
    ax.plot(x_test[:, 0], model.predict(x_test), color="tab:blue", label=f"degree {degree}")
    ax.set_title(f"Degree {degree}")
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
axes[-1].legend(loc="upper right")
fig.tight_layout()
plt.close(fig)

{
    degree: {
        "train_mse": float(mean_squared_error(y_train, model.predict(x_train))),
        "test_mse": float(mean_squared_error(y_test, model.predict(x_test))),
    }
    for degree, model in models.items()
}


## Step 4 - Estimate bias and variance by repeated sampling

The theoretical decomposition concerns repeated datasets. We approximate that experiment by refitting the learner on many fresh samples and tracking predictions over a grid of input points.


In [ ]:
grid = np.linspace(0.0, 1.0, 250).reshape(-1, 1)
signal = target_function(grid[:, 0])
noise_std = 0.22
repetitions = 50
degrees_for_decomposition = [1, 10]
prediction_store = {degree: [] for degree in degrees_for_decomposition}

for rep in range(repetitions):
    sample_seed = SEED + rep + 100
    x_rep, y_rep, _, _ = make_dataset(sample_size=18, noise_std=noise_std, seed=sample_seed)
    for degree in degrees_for_decomposition:
        model = degree_pipeline(degree).fit(x_rep, y_rep)
        prediction_store[degree].append(model.predict(grid))

summary = {}
fig, axes = plt.subplots(1, len(degrees_for_decomposition), figsize=(10.5, 3.8), sharey=True)

for ax, degree in zip(axes, degrees_for_decomposition):
    preds = np.vstack(prediction_store[degree])
    mean_pred = preds.mean(axis=0)
    pointwise_bias_sq = (mean_pred - signal) ** 2
    pointwise_variance = preds.var(axis=0)

    summary[degree] = {
        "estimated_integrated_bias_sq": float(np.trapezoid(pointwise_bias_sq, grid[:, 0])),
        "estimated_integrated_variance": float(np.trapezoid(pointwise_variance, grid[:, 0])),
        "noise_floor": float(noise_std ** 2),
    }

    ax.plot(grid[:, 0], signal, linestyle="--", color="tab:gray", label="true signal")
    ax.plot(grid[:, 0], mean_pred, color="tab:blue", label="average prediction")
    ax.fill_between(
        grid[:, 0],
        mean_pred - preds.std(axis=0),
        mean_pred + preds.std(axis=0),
        color="tab:blue",
        alpha=0.15,
        label="prediction std. dev.",
    )
    ax.set_title(f"Degree {degree}")
    ax.set_xlabel("x")

axes[0].set_ylabel("prediction")
axes[-1].legend(loc="upper right")
fig.tight_layout()
plt.close(fig)

summary


## Interpretation

Typical outcomes are:
- training MSE decreases with degree;
- test and cross-validation MSE usually hit a minimum at an intermediate degree;
- the low-degree model has larger bias and lower variance; and
- the high-degree model has lower bias near the sample points but much larger variance across datasets.

Try changing `sample_size`, `noise_std`, or the maximum degree. Larger datasets usually flatten the variance penalty, while noisier data make overfitting appear sooner.
